## Q1. Cap Stripping Mechanics

### Data Imports

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import display

%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 5)
pd.set_option('display.max_columns', 15)

DATA_PATH = Path('Data')

In [3]:
# Load cap vol time series
cap_raw = pd.read_excel(DATA_PATH / 'project_cap_vol_ts.xlsx', sheet_name='cap', index_col=0)
cap_maturities = cap_raw.loc['maturity'].astype(float)
cap_data = cap_raw.drop(index='maturity').astype(float)
cap_data.index = pd.to_datetime(cap_data.index)

# Round maturities to nearest year (BB reports e.g. 0.999 instead of 1.0)
cap_data.columns = (4 * cap_maturities.values).round(0) / 4
cap_data.columns.name = 'maturity'
cap_data = cap_data.T.drop_duplicates().T

# Load SOFR swap time series
sofr_raw = pd.read_excel(DATA_PATH / 'project_cap_vol_ts.xlsx', sheet_name='sofr', index_col=0)
sofr_maturities = sofr_raw.loc['maturity'].astype(float)
sofr_data = sofr_raw.drop(index='maturity').astype(float)
sofr_data.index = pd.to_datetime(sofr_data.index)
sofr_data.columns = (12 * sofr_maturities.values).round(0) / 12
sofr_data.columns.name = 'maturity'
sofr_data = sofr_data.T.drop_duplicates().T
sofr_data = sofr_data / 100  # percent to decimal

# Load reference rates (SOFR daily)
ref_rates = pd.read_excel(DATA_PATH / 'ref_rates.xlsx', sheet_name='data')
ref_rates['date'] = pd.to_datetime(ref_rates['date'])
ref_rates = ref_rates.set_index('date').sort_index()

sofr_daily = ref_rates['SOFR'].dropna() / 100

# Load validation curves (for 2025-06-30, fully processed)
curves_validation = pd.read_excel(
    DATA_PATH / 'cap_curves_2025-06-30.xlsx',
    sheet_name='rate curves 2025-06-30'
).set_index('tenor')


In [4]:
cap_data.head(5)

maturity,1.0,2.0,3.0,4.0,5.0,6.0,7.0,8.0,9.0,10.0
date,,,,,,,,,,
2022-03-17,127.3,108.5,109.9,108.5,107.3,104.3,101.0,97.7,95.2,93.2
2022-03-18,96.2,104.9,108.0,108.5,108.0,105.0,101.8,98.3,95.4,93.4
2022-03-21,96.2,105.0,108.1,108.5,107.9,105.0,101.7,98.2,95.4,93.4
2022-03-22,75.1,108.3,115.2,115.3,113.6,109.9,106.3,102.8,99.9,97.5
2022-03-23,94.1,111.0,117.9,117.1,114.6,110.7,107.2,103.5,100.5,98.1


In [5]:
sofr_data.head(5)

maturity,0.25,0.50,0.75,1.00,1.25,1.50,1.75,...,4.50,5.00,6.00,7.00,8.00,9.00,10.00
date,,,,,,,,,,,,,,,
2022-01-03,0.000907,0.002001,0.003068,0.004059,0.004970,0.005920,0.006795,...,0.011550,0.012022,0.012636,0.013149,0.013567,0.013908,0.014237
2022-01-04,0.000908,0.001974,0.002980,0.003949,0.004875,0.005808,0.006670,...,0.011560,0.011996,0.012663,0.013228,0.013667,0.014028,0.014370
2022-01-05,0.000982,0.002180,0.003313,0.004400,0.005390,0.006395,0.007355,...,0.012295,0.012746,0.013410,0.013943,0.014349,0.014678,0.014994
2022-01-06,0.001118,0.002390,0.003569,0.004630,0.005655,0.006670,0.007650,...,0.012745,0.013161,0.013794,0.014285,0.014654,0.014946,0.015240
2022-01-07,0.001143,0.002373,0.003505,0.004584,0.005625,0.006660,0.007665,...,0.013020,0.013440,0.014133,0.014665,0.015058,0.015354,0.015648


In [6]:
ref_rates.head(5)

,DTB3,DFF,SOFR
date,,,
2018-01-01,NaN,1.33,NaN
2018-01-02,1.42,1.42,NaN
2018-01-03,1.39,1.42,NaN
2018-01-04,1.39,1.42,NaN
2018-01-05,1.37,1.42,NaN


In [7]:
curves_validation.head(5)

,swap rates,spot rates,discounts,forwards,flat vols,fwd vols
tenor,,,,,,
0.25,0.042353,0.042353,0.989523,NaN,NaN,NaN
0.50,0.040859,0.040852,0.979883,0.039351,0.156842,0.156842
0.75,0.039391,0.039372,0.971043,0.036414,0.180709,0.201708
1.00,0.038115,0.038083,0.962807,0.034217,0.204576,0.240464
1.25,0.036704,0.036653,0.955417,0.030938,0.242127,0.328341


## Question 1: Cap Stripping Mechanics

a). Select a single date from the time series. Using that date’s cap flat vols and SOFR swap curve, build the full processing pipeline:

Convert Bloomberg normal vols (bp) to Black vols

Interpolate to a quarterly grid

Construct discount and forward curves

Bootstrap forward vols from flat vols

Validate your pipeline against the processed ```cap_curves_2025-06-30.xlsx``` file.

b) Plot the flat vol and forward vol term structures side by side. The forward vol curve is typically humped: near-term vol reflects current policy uncertainty, while long-term vol reverts to a mean. Describe the shape you observe and explain why it looks that way given where the Fed was in its policy cycle on your chosen date.

c) Repeat Q1b on 3–4 dates spanning the rate cycle (e.g., mid-2022, mid-2023, mid-2024, mid-2025). How does the shape of the forward vol curve change across regimes?



In [10]:
cap_vols = cap_data.loc['2025-06-30']
SOFR_swaps = sofr_data.loc['2025-06-30']
ref_forward = ref_rates.loc['2025-06-30', 'SOFR']
black_vols = cap_vols / ref_forward
display(black_vols)
display(SOFR_swaps)
display(ref_forward)

maturity
1.0     15.730337
2.0     20.539326
3.0     21.842697
4.0     22.202247
5.0     22.314607
6.0     22.337079
7.0     22.292135
8.0     22.269663
9.0     22.224719
10.0    22.134831
Name: 2025-06-30 00:00:00, dtype: float64

maturity
0.25     0.043019
0.50     0.041467
0.75     0.039991
1.00     0.038667
1.25     0.037205
1.50     0.036121
1.75     0.035409
2.00     0.034903
2.25     0.034435
2.50     0.034177
2.75     0.034045
3.00     0.033993
3.50     0.033879
4.00     0.033967
4.50     0.034055
5.00     0.034284
6.00     0.034770
7.00     0.035305
8.00     0.035846
9.00     0.036364
10.00    0.036865
Name: 2025-06-30 00:00:00, dtype: float64

np.float64(4.45)